# Flash module quickstart

This notebook demonstrates how to import the `flash` module, point it at the bundled component database, and set up both single-component and binary mixtures for simple flash calculations.

In [1]:
# get the project root (one level above ./notebooks)
import sys
import os
root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(root)

from flash import (
    DEFAULT_DATABASE_PATH,
    Mixture,
    NPTEquilibriumSystem,
    PengRobinsonEOS,
    SimpleFlashCalculator,
)

DEFAULT_DATABASE_PATH


PosixPath('/Users/ahmadalkadri/Documents/GitHub/Chemical-Thermodynamics/database/database.h5')

## Single-component feed

Create a mixture with a single component (Methane) and perform a basic flash at 300 K and 1 bar. The default flash calculator assumes a single vapor phase and simply echoes the feed composition.

In [2]:
methane = Mixture.from_names(
    names=["Methane"],
    mole_fractions=[1.0],
    database_path=DEFAULT_DATABASE_PATH,
)

single_component_system = NPTEquilibriumSystem(
    temperature=25+273.15,
    pressure=1.0,
    mixture=methane,
    flash_calculator=SimpleFlashCalculator(eos_model=PengRobinsonEOS()),
)
single_flash = single_component_system.flash()
single_flash.phases["vapor"]

,name,z,MW[g/mol],Tc[K],Pc[bar],omega,Antoine_A,Antoine_B,Antoine_C,Antoine_Tmin[K],...,CpC,CpD,CpE,Cp_Tmin[K],Cp_Tmax[K],phase,Z,Vm[m3/mol],rho[kg/m3],v_specific[m3/kg]
0,Methane,1.0,16.042,190.6,46.0,0.008,8.6041,897.84,-7.16,93.0,...,-0.000002,0.0,0.0,298.0,1500.0,vapor,0.997772,0.024734,0.648571,1.541851


In [3]:
single_flash.specific_properties()

,phase,property,units,value
0,system,temperature,K,298.150000
1,system,pressure,bar,1.000000
2,vapor,z,mole fraction,1.000000
3,vapor,MW[g/mol],g/mol,16.042000
4,vapor,Tc[K],K,190.600000
5,vapor,Pc[bar],bar,46.000000
6,vapor,omega,dimensionless,0.008000
7,vapor,Antoine_A,None,8.604100
8,vapor,Antoine_B,None,897.840000
9,vapor,Antoine_C,None,-7.160000


In [4]:
single_component_system.mixture.components[0].saturation_pressure(100)

0.3441314618297437

In [19]:
c = methane.components[0]

results = [
    c.saturation_pressure(100),
    c.heat_capacity(500)
]

[print(r) for r in results]

0.3441314618297437
47.404908616527


[None, None]

## Additional single-component examples

Compute Peng–Robinson specific properties for a few other pure-component feeds at varied conditions. The results include a ``units`` column alongside each property.

In [5]:
single_component_cases = [
    ("Methane", 300.0, 5.0),
    ("Ethane", 320.0, 8.0),
    ("Propane", 340.0, 10.0),
    ("Methanol", 360.0, 5.0),
]

single_component_properties = {}
for name, T, P in single_component_cases:
    pure_mix = Mixture.from_names(
        names=[name], mole_fractions=[1.0], database_path=DEFAULT_DATABASE_PATH
    )
    pure_system = NPTEquilibriumSystem(
        temperature=T,
        pressure=P,
        mixture=pure_mix,
        flash_calculator=SimpleFlashCalculator(eos_model=PengRobinsonEOS()),
    )
    single_component_properties[name] = pure_system.flash().specific_properties()

import pandas as pd
pd.concat(single_component_properties, names=["component"])


phase           property          units        value
component                                                          
Methane   0   system        temperature              K   300.000000
          1   system           pressure            bar     5.000000
          2    vapor                  z  mole fraction     1.000000
          3    vapor          MW[g/mol]          g/mol    16.042000
          4    vapor              Tc[K]              K   190.600000
...              ...                ...            ...          ...
Methanol  18   vapor         Cp_Tmax[K]              K  1500.000000
          19   vapor                  Z  dimensionless     0.921246
          20   vapor         Vm[m3/mol]         m3/mol     0.005515
          21   vapor         rho[kg/m3]          kg/m3     5.810008
          22   vapor  v_specific[m3/kg]          m3/kg     0.172117

[92 rows x 4 columns]

## Binary feed

Build a two-component mixture (Methane/Ethane) with specified mole fractions and run the same simple flash routine. Attach the Peng–Robinson EOS to compute compressibility, molar volume, density, and specific volume for the vapor phase, then surface every specific property available from the flash calculation.


In [6]:
binary_mix = Mixture.from_names(
    names=["Methane", "Ethane"],
    mole_fractions=[0.4, 0.6],
    database_path=DEFAULT_DATABASE_PATH,
)

binary_flash_calculator = SimpleFlashCalculator(
    eos_model=PengRobinsonEOS(),
)

binary_system = NPTEquilibriumSystem(
    temperature=310.0,
    pressure=5.0,
    mixture=binary_mix,
    flash_calculator=binary_flash_calculator,
)
binary_flash = binary_system.flash()
binary_flash.phases["vapor"][
    ["name", "z", "Z", "Vm[m3/mol]", "rho[kg/m3]", "v_specific[m3/kg]"]
]


,name,z,Z,Vm[m3/mol],rho[kg/m3],v_specific[m3/kg]
0,Methane,0.4,0.97552,0.005029,4.863646,0.205607
1,Ethane,0.6,0.97552,0.005029,4.863646,0.205607


In [7]:
# Model details including EOS configuration
binary_flash.model_details


{'eos': 'Peng-Robinson'}

## Specific property values

Inspect the state variables and phase-level properties computed by the flash calculation. Each row surfaces the property value and its units for quick comparison.

In [8]:
binary_flash.specific_properties()


,phase,property,units,value
0,system,temperature,K,310.0
1,system,pressure,bar,5.0
2,vapor,z (by component),mole fraction,"{'Methane': 0.4, 'Ethane': 0.6}"
3,vapor,MW[g/mol] (by component),g/mol,"{'Methane': 16.042, 'Ethane': 30.069}"
4,vapor,Tc[K] (by component),K,"{'Methane': 190.6, 'Ethane': 305.4}"
5,vapor,Pc[bar] (by component),bar,"{'Methane': 46.0, 'Ethane': 48.74}"
6,vapor,omega (by component),dimensionless,"{'Methane': 0.008, 'Ethane': 0.099}"
7,vapor,Antoine_A (by component),None,"{'Methane': 8.6041, 'Ethane': 9.0435}"
8,vapor,Antoine_B (by component),None,"{'Methane': 897.84, 'Ethane': 1511.42}"
9,vapor,Antoine_C (by component),None,"{'Methane': -7.16, 'Ethane': -17.16}"
